# Installations 

In [1]:
#!pip install datasets
#!pip install torch torchvision transformers faiss-cpu opencv-python scikit-learn

# Step 0: Setup

In [2]:
q_types_mapping = {
    'abnormality_color': 'color',
    'landmark_color': 'color',
    'abnormality_location': 'location',
    'instrument_location': 'location',
    'landmark_location': 'location',
    'finding_count': 'count',
    'instrument_count': 'count',
    'polyp_count': 'count',
    'abnormality_presence': 'yesno',
    'box_artifact_presence': 'yesno',
    'finding_presence': 'yesno',
    'instrument_presence': 'yesno',
    'landmark_presence': 'yesno',
    'text_presence': 'yesno',
    'polyp_removal_status': 'yesno',
    'polyp_type': 'single',
    'polyp_size': 'single',
    'procedure_type': 'single',
}
q_types = ["yesno", "single", "multi", "color", "location", "count"]


In [3]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModel, AutoProcessor, VisionEncoderDecoderModel,
    T5Tokenizer, T5ForConditionalGeneration
)
import torch
import torch.nn as nn

# Load dataset
dataset = load_dataset("SimulaMet/Kvasir-VQA-x1", split="train")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


2025-09-22 15:46:14.535870: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758555974.785836      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758555974.860577      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/143594 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15955 [00:00<?, ? examples/s]

# Step 1: Disease Classification Backbone (TreeNet conversion needed)

In [4]:
!git clone https://github.com/zeshanalvi/feature-extraction.git
import sys
sys.path.append("/kaggle/working/feature-extraction")
from features import get_lires, get_lbps, Dataset, color_layout
import cv2
import numpy as np
import requests

def get_hfimage(img_path):
    response = requests.get(img_path, stream=True)
    if response.status_code == 200:
        img_array = np.asarray(bytearray(response.content), dtype=np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    return img
def get_features(img):
    cl,t1=color_layout(img)
    return cl
def disease_model(img):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.tensor(np.random.rand(23)).to(device)

Cloning into 'feature-extraction'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 135 (delta 56), reused 123 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 24.57 KiB | 4.91 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [5]:
def get_class_all(dataset):
    tensors = []
    for d in dataset:
        tensors.append(disease_model(d['image']))
    return torch.cat(tensors, dim=0)
        
#disease_probs=get_class_all(dataset)

# Step 2: Question-Type Classifier (Router)

In [6]:
router_name = "distilbert-base-uncased"
router_tokenizer = AutoTokenizer.from_pretrained(router_name)
router_model = AutoModelForSequenceClassification.from_pretrained(
    router_name, num_labels=6  # Yes/No, Single, Multi, Color, Location, Count
).to(device)

def classify_question_type(question):
    inputs = router_tokenizer(question, return_tensors="pt", truncation=True).to(device)
    outputs = router_model(**inputs)
    pred = torch.argmax(outputs.logits, dim=1).item()
    return pred


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Step 3: Feature Fusion Layer
Here we define a simple multimodal fusion of:
## Image embedding (ViT-B/16)
## Question embedding (BERT)
## Disease vector (23D)

In [7]:
from transformers import ViTModel, BertModel
import torch.nn.functional as F

image_encoder = ViTModel.from_pretrained("google/vit-base-patch16-224").to(device)
question_encoder = BertModel.from_pretrained("bert-base-uncased").to(device)

class CoAttentionFusion(nn.Module):
    def __init__(self, img_dim, ques_dim, disease_dim, hidden_dim, answer_vocab):
        super(CoAttentionFusion, self).__init__()
        
        self.img_proj = nn.Linear(img_dim, hidden_dim)
        self.ques_proj = nn.Linear(ques_dim, hidden_dim)
        self.dis_proj = nn.Linear(disease_dim, hidden_dim)

        self.att_img = nn.Linear(hidden_dim, 1)
        self.att_dis = nn.Linear(hidden_dim, 1)
        self.fusion = nn.Linear(hidden_dim * 3, hidden_dim)

        # ✅ Store answer vocab inside model for later use
        self.answer_vocab = answer_vocab

    def forward(self, img_feat, ques_feat, dis_vec):
        # Project features
        #print("Input Shapes\t",img_feat.shape, ques_feat.shape, dis_vec.shape)
        img_proj = torch.tanh(self.img_proj(img_feat))     # [B, H]
        ques_proj = torch.tanh(self.ques_proj(ques_feat))  # [B, H]

        print("Ques_proj",ques_proj.shape,img_proj.shape)
        
        dis_vec = dis_vec.to(torch.float32)
        dis_proj = torch.tanh(self.dis_proj(dis_vec))      # [B, H]


        #print("After projection\t",img_proj.shape, ques_proj.shape, dis_proj.shape)

        # Expand question for image alignment
        #ques_expand = ques_proj.unsqueeze(1).expand_as(img_proj)
        
        
        #ques_expand = ques_proj#.expand_as(img_proj)
        #img_co = img_proj * ques_expand

        #Replacement of above 2 lines

        ques_proj = ques_proj.unsqueeze(1)                    # [16, 1, 512]
        ques_expand = ques_proj.expand(-1, img_proj.size(1), -1)  # [16, 197, 512]
        img_co = img_proj * ques_expand                       # [16, 197, 512]

        #print("ques_expand",ques_expand.shape,img_co.shape)
        
        att_img_weights = torch.sigmoid(self.att_img(img_co))  # [B, 1]
        #img_att = att_img_weights * img_proj
        img_att = (att_img_weights * img_proj).sum(1)
        #att_img_weights = F.softmax(self.att_img(img_co), dim=1)   # [B, R, 1]
        #img_att = (att_img_weights * img_proj).sum(1)              # [B, H]

        # Co-attention with disease vector
        dis_co = dis_proj * ques_proj
        att_dis_weights = torch.sigmoid(self.att_dis(dis_co))      # [B, 1]
        dis_att = att_dis_weights * dis_proj                       # [B, H]


        #print(img_att.shape, ques_proj.shape, dis_att.shape)


        if img_att.dim() == 1:
            img_att = img_att.unsqueeze(0)   # [1, H]
        if ques_proj.dim() == 1:
            ques_proj = ques_proj.unsqueeze(0)     # [1, H]
        if dis_att.dim() == 1:
            dis_att = dis_att.unsqueeze(0)     # [1, H]


        # Concatenate

        ques_proj_flat = ques_proj.squeeze(1)
        dis_att_flat = dis_att.squeeze(1) 

        #print(img_att.shape,ques_proj.shape,dis_att.shape)
        
        joint_feat = torch.cat([img_att, ques_proj_flat, dis_att_flat], dim=1)  # [B, 3H]
        fused = torch.tanh(self.fusion(joint_feat))  # [B, H]
        return fused

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTModel were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

# Step 4: Task-Specific Predictors
For clarity, I’ll define placeholders (can be trained separately).

In [8]:
# ---------------------------
# Step 5: Task-Specific Predictors
# ---------------------------
class TaskPredictor(nn.Module):
    def __init__(self, task_type, hidden=512):
        super().__init__()
        if task_type == "yesno":
            self.head = nn.Linear(hidden, 2)
        elif task_type == "single":
            self.head = nn.Linear(hidden, 10)
        elif task_type == "multi":
            self.head = nn.Linear(hidden, 10)
        elif task_type == "color":
            self.head = nn.Linear(hidden, 5)
        elif task_type == "location":
            self.head = nn.Linear(hidden, 6)
        elif task_type == "count":
            self.head = nn.Linear(hidden, 1)
        else:
            raise ValueError("Unknown task")
    
    def forward(self, x):
        return self.head(x)

# Step 5: Descriptive Answer Generator

We’ll use T5 as the main generator (other tasks can swap in BART/GPT-2 as needed).

In [9]:
gen_name = "t5-base"
gen_tokenizer = T5Tokenizer.from_pretrained(gen_name)
gen_model = T5ForConditionalGeneration.from_pretrained(gen_name).to(device)

def generate_descriptive_answer(question, prediction, fused_features):
    # Construct a prompt combining prediction and context
    prompt = f"Question: {question} | Prediction: {prediction} | Context: GI disease analysis"
    inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    outputs = gen_model.generate(**inputs, max_length=50)
    return gen_tokenizer.decode(outputs[0], skip_special_tokens=True)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
class QuestionTypeClassifier(nn.Module):
    def __init__(self, hidden=768, num_types=len(q_types)):
        super().__init__()
        self.fc = nn.Linear(hidden, num_types)

    def forward(self, q_feat):
        return self.fc(q_feat)  # logits [B, 6]
        
qtype_classifier=QuestionTypeClassifier().to(device)

# Step 6: Full Pipeline Integration

# Dataset Downloader

In [11]:
def vqa_pipeline(image, question):
    # 1. Disease classification
    disease_vec = disease_model(image)
    #disease_vec = disease_model(image.unsqueeze(0).to(device))  # (1,23)

    # 2. Question type classification
    q_type_idx = classify_question_type(question)
    q_types = ["yesno", "single", "multi", "color", "location", "count"]
    task_type = q_types[q_type_idx]

    # 3. Image + Question embeddings
    img_emb = image_encoder(pixel_values=image.unsqueeze(0).to(device)).pooler_output
    q_inputs = router_tokenizer(question, return_tensors="pt", truncation=True).to(device)
    q_emb = question_encoder(**q_inputs).pooler_output

    # 4. Fusion
    img_emb = img_emb.float()
    q_emb = q_emb.float()
    disease_vec = disease_vec.float()
    dis_vec = disease_vec.unsqueeze(0).float().to(device)
    #print("Image\t",img_emb.shape,img_emb.device)
    #print("Question\t",q_emb.shape,q_emb.device)
    #print("Disease\t",dis_vec.shape,dis_vec.device)
    
    fusion_module = CoAttentionFusion(img_dim=768, ques_dim=768, disease_dim=23, hidden_dim=512).to(device)
    fused=fusion_module.forward(img_emb, q_emb, dis_vec)
    #fused = FusionModule()(img_emb, q_emb, disease_vec)

    # 5. Task-specific predictor
    predictor = TaskPredictor(task_type).to(device)
    pred_out = predictor(fused)

    #print("Prediction\t",pred_out.shape,pred_out)
    print("Task Type\t",task_type)

    # Convert prediction to text for generator
    if task_type == "yesno":
        pred_label = "Yes" if torch.argmax(pred_out) == 1 else "No"
    elif task_type == "count":
        pred_label = f"{pred_out.item():.0f}"
    else:
        pred_label = str(torch.argmax(pred_out).item())

    print("Label\t",pred_label)
    # 6. Generate descriptive answer
    answer = generate_descriptive_answer(question, pred_label, fused)
    return answer


# Read Dataset images download

In [12]:
import requests
from PIL import Image
from io import BytesIO
import torchvision.transforms as transforms
import torch

def read_image(img_path):
    response = requests.get(img_path)
    img = Image.open(BytesIO(response.content)).convert("RGB")
    # Step 2: Define transforms (resize, convert to tensor, normalize, etc.)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),   # adjust size for your model
        transforms.ToTensor(),           # convert to tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                             std=[0.229, 0.224, 0.225])
    ])
    # Step 3: Apply transforms
    image = transform(img)
    # Step 4: Add batch dimension and move to device
    #device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    #image1 = image.unsqueeze(0).to(device)
    return image


def prepare_dataset():
    return 


#print(image1.shape)  # should be [1, 3, 224, 224]



# Data Preparation

In [13]:
from datasets import load_dataset
#ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")
ds_train = load_dataset("SimulaMet/Kvasir-VQA-x1", split="train")
ds_test = load_dataset("SimulaMet/Kvasir-VQA-x1", split="test")


In [14]:
from PIL import Image
import requests
import torch
import torchvision.transforms as transforms
transformten = transforms.Compose([
        transforms.Resize((224, 224)),   # adjust size for your model
        transforms.ToTensor(),           # convert to tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet normalization
                             std=[0.229, 0.224, 0.225])
    ])



def preprocess_example(example):
    # Download image
    image = Image.open(requests.get(example["image"], stream=True).raw).convert("RGB")
    
    # Apply your normalize/transform method
    image = transformten(image)  # e.g. Resize + ToTensor + Normalize


    #print("DEBUG image:", type(image), image.shape)

    # Tokenize the question
    q_inputs = router_tokenizer(example["question"], 
                                return_tensors="pt", 
                                truncation=True, 
                                padding="max_length", 
                                max_length=32)

    # q_inputs is a BatchEncoding with tensors inside (batch_size=1), so we squeeze
    input_ids = q_inputs["input_ids"].squeeze(0)          # torch.Tensor [seq_len]
    attention_mask = q_inputs["attention_mask"].squeeze(0)
    
    # Pack features
    return {
        "image": image,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "answer": example["answer"],
        "question_class": example["question_class"],
        "image_url": example["image"],
    }


split_dataset_dict = ds_train.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset_dict['train'].select(range(100))
val_dataset = split_dataset_dict['test'].select(range(100))


#print(train_dataset)
#print(val_dataset)

train_data = train_dataset.map(preprocess_example)
val_data = val_dataset.map(preprocess_example)

print(train_data[0].keys())
print(type(train_data[0]['image']))

# Tell HF to keep tensors
#train_data.set_format(type="torch", columns=["image"])
#val_data.set_format(type="torch", columns=["image"])


print(train_data[0].keys())
print(type(train_data[0]['image']))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

dict_keys(['image', 'complexity', 'question', 'answer', 'original', 'question_class', 'img_id', 'input_ids', 'attention_mask', 'image_url'])
<class 'list'>
dict_keys(['image', 'complexity', 'question', 'answer', 'original', 'question_class', 'img_id', 'input_ids', 'attention_mask', 'image_url'])
<class 'list'>


In [15]:
def normalize_answer(ans, q_type):
    ans = ans.strip().lower()

    if q_type == "yesno":
        if "yes" in ans or "present" in ans or "evidence" in ans:
            return "Yes"
        elif "no" in ans or "absent" in ans or "none" in ans:
            return "No"
        else:
            return None  # ambiguous

    if q_type == "count":
        # Extract numeric value or return None
        from re import findall
        numbers = findall(r"\d+", ans)
        if numbers:
            return numbers[0]
        elif "one" in ans: return "1"
        elif "two" in ans: return "2"
        return None

    if q_type == "color":
        for color in ["red","green","yellow","blue","white","black"]:
            if color in ans:
                return color
        return None

    if q_type == "location":
        # Simplify locations to a small fixed set
        for loc in ["upper","lower","left","right","central"]:
            if loc in ans:
                return loc
        return None

    if q_type in ["single","multi"]:
        return ans  # keep original but can also restrict choices

    return ans


def build_vocabs(dataset):
    # Build task-specific vocabularies
    task_vocabs = {}
    for general_class in set(q_types_mapping.values()):
        task_vocabs[general_class] = {}
    
    for row in dataset:
        fine_class = row["question_class"]

        # ✅ Handle if fine_class is a list
        if isinstance(fine_class, list):
            fine_class = fine_class[0]  

        general_class = q_types_mapping[fine_class]

        norm_ans = normalize_answer(row["answer"], general_class)
        if norm_ans is None:
            continue  # skip unnormalizable answers

        if norm_ans not in task_vocabs[general_class]:
            idx = len(task_vocabs[general_class])
            task_vocabs[general_class][norm_ans] = idx

    return task_vocabs

task_vocabs=build_vocabs(train_data)
#print(task_vocabs)


from collections import defaultdict

def build_answer_vocab(dataset, q_types_mapping):
    answer_vocab = defaultdict(dict)
    counters = defaultdict(int)

    for ans, q_class in zip(dataset["answer"], dataset["question_class"]):
        # q_class might be a list; pick the first (if multiple labels)
        if isinstance(q_class, list):
            q_class = q_class[0]

        general_class = q_types_mapping[q_class]

        if ans not in answer_vocab[general_class]:
            answer_vocab[general_class][ans] = counters[general_class]
            counters[general_class] += 1

    return answer_vocab


# Example usage:
# dataset = load_dataset("SimulaMet/Kvasir-VQA-x1", split="train")
answer_vocabs = build_answer_vocab(train_data, q_types_mapping)

In [16]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    print(type(batch[0]["image"]))
    
    #images = torch.stack([item["image"] for item in batch])
    images = torch.stack([torch.tensor(item["image"]) if isinstance(item["image"], list) else item["image"] for item in batch])
    
    print(type(images), images.shape)


    input_ids = torch.stack([torch.tensor(item["input_ids"]) if isinstance(item["input_ids"], list) else item["input_ids"] for item in batch])
    attention_mask = torch.stack([torch.tensor(item["attention_mask"]) if isinstance(item["attention_mask"], list) else item["attention_mask"] for item in batch])


    
    #input_ids = torch.stack([item["input_ids"] for item in batch])
    #attention_mask = torch.stack([item["attention_mask"] for item in batch])
    answers = [item["answer"] for item in batch]  # keep as list for label encoding later
    q_classes = [item["question_class"] for item in batch]
    return {
        "images": images,
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "answers": answers,
        "question_classes": q_classes,
    }

train_loader = DataLoader(train_data, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False, collate_fn=collate_fn)


# Training

## Forward

## Loss function

In [17]:
fusion_module = CoAttentionFusion(img_dim=768, ques_dim=768,
                                  disease_dim=23, hidden_dim=512, answer_vocab=answer_vocabs).to(device)

def forward_batch(images, input_ids, attention_mask, answers, true_q_classes=None):
    # Disease vector (dummy placeholder: replace with your trained disease model)
    disease_vec = disease_model(images)  # [B, 23]

    # Encode image
    img_outputs = image_encoder(pixel_values=images.to(device))
    img_feat = img_outputs.last_hidden_state  # [B, R, 768]

    # Encode question
    q_feat = question_encoder(input_ids=input_ids.to(device),
                              attention_mask=attention_mask.to(device)).pooler_output  # [B, 768]

    # Predict task type from question
    print(q_feat.device)
    task_logits = qtype_classifier(q_feat)  # [B, 6]
    task_pred = torch.argmax(task_logits, dim=1)  # predicted type index
    
    # Fusion
    fused = fusion_module(img_feat, q_feat, disease_vec)

    # Task-specific predictions
    preds = []
    for i, t_idx in enumerate(task_pred):
        task_type = q_types[t_idx]  # map index to string
        predictor = TaskPredictor(task_type).to(device)
        preds.append(predictor(fused[i].unsqueeze(0)))

    return preds, answers, task_logits
    
    #for i, task_type in enumerate(q_classes):
    #    predictor = TaskPredictor(task_type).to(device)
    #    pred_out = predictor(fused[i].unsqueeze(0))
    #    preds.append(pred_out)
    #return preds, answers


In [18]:
from torch.nn import CrossEntropyLoss, MSELoss

# Example: classification for yes/no/single/multi/color/location
ce_loss = CrossEntropyLoss()
mse_loss = MSELoss()

import re

def extract_count(answer_str):
    """
    Try to convert an answer string into a number.
    Returns None if it cannot be parsed.
    """
    try:
        # Direct numeric
        return float(answer_str)
    except ValueError:
        pass

    # Handle words like "one", "two", etc.
    word2num = {
        "zero": 0, "one": 1, "two": 2, "three": 3,
        "four": 4, "five": 5, "six": 6,
        "seven": 7, "eight": 8, "nine": 9, "ten": 10
    }
    tokens = answer_str.lower().split()
    for t in tokens:
        if t in word2num:
            return float(word2num[t])

    # Extract any digits from the string
    numbers = re.findall(r"\d+", answer_str)
    if numbers:
        return float(numbers[0])

    return None  # fallback

def compute_loss1(preds, answers, task_logits, true_q_classes):
    total_loss = 0.0

    mapped_classes = [q_types_mapping[c[0] if isinstance(c, list) else c] 
                      for c in true_q_classes]

    # Loss 1: question type classification
    true_task_types = torch.tensor(
        [q_types.index(c) for c in mapped_classes],
        device=task_logits.device
    )
    task_loss = ce_loss(task_logits, true_task_types)
    total_loss += task_loss

    # Loss 2: answer prediction
    for pred, ans, c in zip(preds, answers, mapped_classes):
        if c == "count":
            ans_val = extract_count(ans)
            if ans_val is None:
                print(f"[Warning] Could not parse count answer: {ans}")
                continue
            ans_val = torch.tensor([ans_val], dtype=torch.float, device=pred.device)
            total_loss += mse_loss(pred.squeeze(), ans_val)

        else:
            if ans not in answer_vocab[c]:
                continue
            ans_idx = answer_vocab[c][ans]
            ans_tensor = torch.tensor([ans_idx], dtype=torch.long, device=pred.device)

            if ans_idx >= pred.size(1):
                print(f"[Warning] Skipping answer {ans} for task {c}: index {ans_idx} >= pred.size(1)")
                continue

            total_loss += ce_loss(pred, ans_tensor)

    return total_loss / len(preds)



In [19]:
def compute_loss11(preds, answers, task_logits, true_q_classes, answer_vocab, q_types_mapping, q_types):
    total_loss = 0.0

    # Step 1. Map fine-grained → general
    mapped_classes = [
        q_types_mapping[c[0] if isinstance(c, list) else c] 
        for c in true_q_classes
    ]

    # Step 2. Question type classification loss
    true_task_types = torch.tensor(
        [q_types.index(c) for c in mapped_classes],
        device=task_logits.device
    )
    task_loss = ce_loss(task_logits, true_task_types)
    total_loss += task_loss

    # Step 3. Answer prediction loss
    for pred, ans, c in zip(preds, answers, mapped_classes):
        if c == "count":
            # Normalize numeric answer
            ans_val = normalize_answer(ans, c)
            if ans_val is None:
                print(f"[Warning] Could not parse count answer: {ans}")
                continue
            ans_val = torch.tensor([float(ans_val)], dtype=torch.float, device=pred.device)
            total_loss += mse_loss(pred.squeeze(), ans_val)

        else:
            # Normalize categorical answer
            norm_ans = normalize_answer(ans, c)
            if norm_ans is None or norm_ans not in answer_vocab[c]:
                print(f"[Warning] Skipping unrecognized answer: {ans} (task {c})")
                continue

            ans_idx = answer_vocab[c][norm_ans]
            ans_tensor = torch.tensor([ans_idx], dtype=torch.long, device=pred.device)

            if ans_idx >= pred.size(1):
                print(f"[Warning] Skipping answer {ans} (normalized {norm_ans}) for task {c}: "
                      f"index {ans_idx} >= pred.size(1)={pred.size(1)}")
                continue

            total_loss += ce_loss(pred, ans_tensor)

    return total_loss / len(preds)

In [20]:
def compute_loss(preds, answers, task_logits, true_q_classes, answer_vocabs):
    """
    preds: list of model predictions for each sample
    answers: list of strings (descriptive answers)
    task_logits: tensor [batch_size, num_task_types]
    true_q_classes: list of lists (fine-grained classes for each question)
    answer_vocabs: dict mapping {q_type: {answer: index}}
    """
    total_loss = 0

    # 1) Map fine-grained → general classes
    mapped_classes = [
        q_types_mapping[c[0] if isinstance(c, list) else c]
        for c in true_q_classes
    ]

    # 2) Question type classification loss
    true_task_types = torch.tensor(
        [q_types.index(c) for c in mapped_classes],
        device=task_logits.device
    )
    task_loss = ce_loss(task_logits, true_task_types)
    total_loss += task_loss

    # 3) Answer prediction loss (per sample)
    for pred, ans, c in zip(preds, answers, mapped_classes):
        if c == "count":
            # For count, answer must be numeric
            try:
                ans_val = float(ans)
                ans_val = torch.tensor([ans_val], device=pred.device)
                total_loss += mse_loss(pred.squeeze(), ans_val)
            except ValueError:
                print(f"[Warning] Skipping non-numeric count answer: {ans}")
                continue

        else:
            # For categorical tasks (yesno, single, multi, etc.)
            if ans not in answer_vocabs.get(c, {}):
                print(f"[Warning] Skipping unseen or descriptive answer {ans} for task {c}")
                continue

            ans_idx = answer_vocabs[c][ans]

            if ans_idx >= pred.size(1):
                print(f"[Warning] Skipping answer {ans} for task {c}: "
                      f"index {ans_idx} >= pred.size(1)")
                continue

            ans_tensor = torch.tensor([ans_idx], device=pred.device)
            total_loss += ce_loss(pred, ans_tensor)

    return total_loss / len(preds)

## Training Loop

In [21]:
epochs=10

fusion_module = CoAttentionFusion(img_dim=768, ques_dim=768, 
                                  disease_dim=23, hidden_dim=512,
                                  answer_vocab=answer_vocabs).to(device)


optimizer = torch.optim.AdamW(list(fusion_module.parameters()) + 
                              list(question_encoder.parameters()) + 
                              list(image_encoder.parameters())+
                              list(qtype_classifier.parameters()), lr=2e-5)

for epoch in range(10):
    fusion_module.train()
    qtype_classifier.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        preds, answers, task_logits = forward_batch(
            batch["images"],
            batch["input_ids"],
            batch["attention_mask"],
            batch["answers"],
            batch["question_classes"]  # fine-grained from dataset
        )
        #preds, answers = forward_batch(batch["images"],batch["input_ids"], batch["attention_mask"], batch["answers"], batch["question_classes"])
        loss = compute_loss(preds, answers, task_logits, batch["question_classes"],answer_vocabs=answer_vocabs)
        #loss = compute_loss(preds, answers, batch["question_classes"])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch}, Train Loss: {total_loss / len(train_loader)}")


<class 'list'>
<class 'torch.Tensor'> torch.Size([16, 3, 224, 224])
cuda:0
Ques_proj torch.Size([16, 512]) torch.Size([16, 197, 512])
[Warning] Skipping answer No residual polyp tissue identified for task yesno: index 16 >= pred.size(1)
[Warning] Skipping answer Evidence of a single polyp remaining in the visualized area. for task yesno: index 11 >= pred.size(1)
[Warning] Skipping answer No residual polyps identified and no green or black box artifacts present for task yesno: index 23 >= pred.size(1)
[Warning] Skipping answer Instrument visible in central and lower regions with biopsy forceps noted for task location: index 12 >= pred.size(1)
[Warning] Skipping answer No residual polyps identified, one instrument visualized for task yesno: index 32 >= pred.size(1)
[Warning] Skipping answer No identifiable anatomical landmark observed for task location: index 11 >= pred.size(1)
[Warning] Skipping answer Abnormalities are distributed across multiple regions including central, upper, and l

## Validation Loop

In [22]:
fusion_module.eval()
with torch.no_grad():
    total_loss = 0
    for batch in val_loader:
        preds, answers, task_logits = forward_batch(
            batch["images"],
            batch["input_ids"],
            batch["attention_mask"],
            batch["answers"],
            batch["question_classes"]
        )

        loss = compute_loss(
            preds,
            answers,
            task_logits,
            batch["question_classes"],
            answer_vocabs
        )
        total_loss += loss.item()
    print(f"Validation Loss: {total_loss / len(val_loader)}")


<class 'list'>
<class 'torch.Tensor'> torch.Size([16, 3, 224, 224])
cuda:0
Ques_proj torch.Size([16, 512]) torch.Size([16, 197, 512])
[Warning] Skipping non-numeric count answer: No surgical instruments observed and text is present
[Warning] Skipping unseen or descriptive answer Abnormalities show pink and red hues with no visible text and no identifiable anatomical landmarks. for task color
[Warning] Skipping unseen or descriptive answer Evidence of a green/black box artifact is present, with red and pink abnormalities noted, and a single polyp observed. for task yesno
[Warning] Skipping unseen or descriptive answer gastroscopic examination performed for task single
[Warning] Skipping answer No evidence of box artifacts observed for task yesno: index 31 >= pred.size(1)
[Warning] Skipping unseen or descriptive answer No green or black box artifacts observed; multiple abnormalities noted in pink and white hues for task yesno
[Warning] Skipping unseen or descriptive answer polyp measurin

In [23]:
#from transformers import BlipProcessor



#processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

def preprocess(example):
    inputs = processor(images=example['image'], text=example['question'], return_tensors="pt", padding="max_length", truncation=True)
    inputs['labels'] = processor.tokenizer(example['answer'], padding="max_length", truncation=True, return_tensors="pt").input_ids
    return inputs


def preprocess(batch):
    images = []
    for img in batch["image"]:
        if isinstance(img, str):  # URL string
            response = requests.get(img)
            image = Image.open(BytesIO(response.content)).convert("RGB")
        else:
            image = img
        images.append(image)

    # Processor handles lists of images + texts
    inputs = processor(
        images=images,
        text=batch["question"],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
    )

    labels = processor.tokenizer(
        batch["answer"],
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).input_ids

    inputs["labels"] = labels
    return inputs



#res=preprocess(ds_train[1])
#print(res["labels"])
#train_dataset = ds_train.map(preprocess, batched=True, remove_columns=ds_train.column_names)
#print(train_dataset)

In [24]:
#from transformers import BlipForConditionalGeneration, Trainer, TrainingArguments

#model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")


def comments():
    
    training_args = TrainingArguments(
        per_device_train_batch_size=4,
        num_train_epochs=5,
        evaluation_strategy="no",
        logging_dir="./logs",
        output_dir="./blip-medical-captioning",
        save_strategy="epoch"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        tokenizer=processor.tokenizer
    )
    
    trainer.train()


In [25]:
#image = dataset['raw'][0]['image']
#inputs = processor(images=image, return_tensors="pt").to(model.device)
#output = model.generate(**inputs)
#caption = processor.tokenizer.decode(output[0], skip_special_tokens=True)
#print("Generated Caption:", caption)
